# Feature Selection Experiment Results

## Objective

Evaluate the impact of applying **feature selection** using **SelectFdr** on the best preprocessing pipeline.

## Results

| Model | Feature Selection | Test AP | Train AP | Validation AP |
|------|-------------------|--------:|---------:|--------------:|
| Gradient Boost + Random Over Sampler | FDR (α = 0.2) | 0.101517 | 0.118274 | **0.086280** |
| Logistic Regression (Elastic Net) + Baseline | None | 0.107042 | 0.086293 | 0.086080 |
| Logistic Regression + Baseline | FDR (α = 0.1) | **0.108235** | 0.086116 | 0.085976 |
| Logistic Regression (Lasso) + Baseline | None | 0.105340 | 0.086865 | 0.085889 |
| Logistic Regression + Baseline | FDR (α = 0.4) | 0.105116 | 0.086510 | 0.085885 |
| Logistic Regression (Ridge) + Baseline | None | 0.106976 | 0.086522 | 0.085879 |
| Logistic Regression (Lasso) + Baseline | FDR (α = 0.1) | 0.105050 | 0.086453 | 0.085876 |
| Gradient Boost + Random Over Sampler | None | 0.104124 | 0.115940 | 0.085872 |
| Logistic Regression (Elastic Net) + Baseline | FDR (α = 0.4) | 0.107237 | 0.085928 | 0.085851 |
| Gradient Boost + Random Over Sampler | FDR (α = 0.3) | 0.102077 | 0.119205 | 0.085804 |

## Conclusion

- Applying **SelectFdr (α = 0.2)** with **Gradient Boost + Random Over Sampler** achieved the highest **Validation Average Precision (0.086280)**.
- However, the gap between **Train AP (0.118274)** and **Validation AP (0.086280)** became noticeably larger, indicating increased overfitting.
- Considering both **Validation Average Precision** and **generalization performance**, **Logistic Regression (Elastic Net) without feature selection** remains the preferred model due to its comparable validation performance and substantially smaller train-validation gap.

In [105]:
import warnings
warnings.filterwarnings("ignore")

In [106]:
from pathlib import Path
import sys

project_root = Path.cwd().parents[1]
sys.path.insert(0, str(project_root))

In [107]:
READ_CSV="../../data/interim/data_travel_insurance_interim.csv"
SAVE_RESULT = "results/hyperparameter_tuning.csv"
RANDOM_STATE=42


TARGET_TRANSFORM_COLS = ["Destination"]
LOG_TRANSFORM_COLS= ["Duration", "Net Sales"]

In [108]:
from src.utils import benchmark_models

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder, TargetEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import PowerTransformer
from sklearn.feature_selection import SelectFdr, SelectFromModel, f_classif
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier

from feature_engine.outliers import Winsorizer

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline

In [109]:
df = pd.read_csv(READ_CSV)
df.head()

,Agency,Agency Type,Distribution Channel,Product Name,Duration,Destination,Net Sales,Commission,Age,Claim,Is Refund,Suspected Fraud,Commission Rate
0,C2B,Airlines,Online,Annual Silver Plan,365,SINGAPORE,216.0,54.0,57,0,No,No,0.25
1,EPX,Travel Agency,Online,Cancellation Plan,4,MALAYSIA,10.0,0.0,33,0,No,No,0.00
2,JZI,Airlines,Online,Basic Plan,19,INDIA,22.0,7.7,26,0,No,No,0.35
3,EPX,Travel Agency,Online,2 way Comprehensive Plan,20,UNITED STATES,112.0,0.0,59,0,No,No,0.00
4,C2B,Airlines,Online,Bronze Plan,8,SINGAPORE,16.0,4.0,28,0,No,No,0.25


In [110]:
x = df.drop(columns=["Claim"])
y = df["Claim"]


In [111]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

In [112]:
NUMERIC_COLS = [features for features in x_train.columns if x_train[features].dtypes != "O"]
CATEGORICAL_COLS = [features for features in x_train.columns if x_train[features].dtypes == "O"]


In [113]:
numeric_pipeline = Pipeline([
    ("winsorizer_iqr", Winsorizer(capping_method="iqr", fold=1.5)),
    ("RobustScaler", RobustScaler()),
])

numeric_log_pipeline = Pipeline([
    ("power", PowerTransformer(method="yeo-johnson")),
    ("RobustScaler", RobustScaler()),
])

categorical_ohe_pipeline = Pipeline([
     ("OneHotEncoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first"))
])

categorical_target_pipeline = Pipeline([
     ("TargetEncoder", TargetEncoder())
])

In [114]:
preprocessor = ColumnTransformer(
    [
       
        ("numeric_pipeline", numeric_pipeline, [c for c in NUMERIC_COLS if c not in LOG_TRANSFORM_COLS]),
        ("numeric_log_pipeline", numeric_log_pipeline, LOG_TRANSFORM_COLS),
        ("categorical_ohe_pipeline", categorical_ohe_pipeline, [c for c in CATEGORICAL_COLS if c not in TARGET_TRANSFORM_COLS]),
        ("categorical_target_pipeline", categorical_target_pipeline, TARGET_TRANSFORM_COLS),

    ],
    remainder="drop"
)

In [115]:
from scipy.stats import randint, uniform

experiments = {
    "GradientBoost": {
        "pipeline": ImbPipeline([
            ("preprocessor", preprocessor),
            ("feature_selection", SelectFdr(score_func=f_classif, alpha=0.005)),
            ("resampler", RandomOverSampler(random_state=RANDOM_STATE)),
            ("classifier", GradientBoostingClassifier(random_state=RANDOM_STATE))
        ]),
        "params": {
            # Slight reduction in maximum trees to prevent late-stage overfitting
            "classifier__n_estimators": randint(50, 300),
            # Paired with lower learning rates for stable step-wise correction
            "classifier__learning_rate": uniform(0.01, 0.15),
            # Keeping depths shallow (2-4) since tree-boosting thrives on weak learners
            "classifier__max_depth": randint(2, 5),
            # Significantly bumped up to prevent isolates from creating custom splits
            "classifier__min_samples_split": randint(15, 60),
            "classifier__min_samples_leaf": randint(10, 40),
            # Row subsampling ensures trees don't see the exact same samples
            "classifier__subsample": uniform(0.6, 0.35),  # range: [0.6, 0.95]
            "classifier__max_features": ["sqrt", "log2"]  # Forcing column feature randomness
        }
    },

    "LogisticRegressionLasso": {
        "pipeline": ImbPipeline([
            ("preprocessor", preprocessor),
            ("feature_selection", SelectFdr(score_func=f_classif, alpha=0.05)),
            ("resampler", "passthrough"),
            ("classifier", LogisticRegression(
                penalty="l1",
                solver="liblinear",
                random_state=RANDOM_STATE
            ))
        ]),
        "params": {
            # Narrowed range: your model generalizes beautifully (0.086 vs 0.086).
            # A C up to 20 risks turning off the L1 penalty completely. C capped at 5 preserves sparsity.
            "classifier__C": uniform(0.001, 5.0), 
            "classifier__fit_intercept": [True, False],
            "classifier__class_weight": [None, "balanced"]
        }
    },

    "RandomForest": {
        "pipeline": ImbPipeline([
            ("preprocessor", preprocessor),
            ("feature_selection",
             SelectFromModel(
                 GradientBoostingClassifier(random_state=RANDOM_STATE),
                 threshold="1.5*mean"
             )),
            ("resampler", "passthrough"),
            ("classifier", RandomForestClassifier(random_state=RANDOM_STATE))
        ]),
        "params": {
            # Compressed from 800 to avoid excess estimators memorizing noise
            "classifier__n_estimators": randint(100, 400),
            # CRITICAL FIX: Removed "None". Uncapped max_depth caused the 0.73 vs 0.05 crash.
            # Explicitly limiting depth forces individual trees to stop early.
            "classifier__max_depth": [4, 6, 9, 12],
            # Drastically higher constraints to enforce massive grouping at terminal leaves
            "classifier__min_samples_split": randint(20, 80),
            "classifier__min_samples_leaf": randint(15, 50),
            "classifier__max_features": ["sqrt", "log2"],
            "classifier__bootstrap": [True], # Forcing bootstrap to remain True to ensure bag variance
            # Added a slight cost-complexity pruning boundary as an extra guardrail
            "classifier__ccp_alpha": uniform(0.0, 0.005) 
        }
    },

    "XGBoost": {
        "pipeline": ImbPipeline([
            ("preprocessor", preprocessor),
            ("feature_selection",
             SelectFromModel(
                 LogisticRegression(
                     penalty="l1",
                     solver="liblinear",
                     random_state=RANDOM_STATE
                 ),
                 threshold="1.5*mean"
             )),
            ("resampler", RandomOverSampler(random_state=RANDOM_STATE)),
            ("classifier",
             XGBClassifier(
                 random_state=RANDOM_STATE,
                 eval_metric="logloss"
             ))
        ]),
        "params": {
            "classifier__n_estimators": randint(100, 400),
            "classifier__learning_rate": uniform(0.01, 0.15),
            # Capping depth at 6; tabular gradient boosting rarely needs >6 depth without overfitting
            "classifier__max_depth": randint(3, 7),
            "classifier__subsample": uniform(0.6, 0.35),     # Subsample rows [0.6, 0.95]
            "classifier__colsample_bytree": uniform(0.5, 0.45), # Subsample columns [0.5, 0.95]
            # Gamma represents the minimum loss reduction required to make a split
            "classifier__gamma": uniform(0.5, 4.5), 
            # High values prevent the algorithm from creating leaves with tiny sum of weights
            "classifier__min_child_weight": randint(5, 25),
            # L1 (Alpha) and L2 (Lambda) regularizations to push leaf weights closer to zero
            "classifier__reg_alpha": uniform(0.1, 4.0),
            "classifier__reg_lambda": uniform(1.0, 5.0)
        }
    }
}

In [ ]:
results = []

for model_name, exp in experiments.items():
    search = RandomizedSearchCV(
        estimator=exp["pipeline"],
        param_distributions=exp["params"],
        n_iter=100,
        scoring="average_precision",
        cv=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
        return_train_score=True
    )

    search.fit(x_train, y_train)

    best_model = search.best_estimator_

    train_prob = best_model.predict_proba(x_train)[:, 1]
    test_prob = best_model.predict_proba(x_test)[:, 1]

    results.append({
        "Model": model_name,
        "ap_validate_score": search.best_score_,
        "ap_train_score": average_precision_score(y_train, train_prob),
        "ap_test_score": average_precision_score(y_test, test_prob),
        "Best Params": search.best_params_
    })
    
df_results = pd.DataFrame(results)

In [117]:
df_results = pd.DataFrame(results).sort_values("ap_validate_score", ascending=False)
df_results

,Model,ap_validate_score,ap_train_score,ap_test_score,Best Params
0,GradientBoost,0.095786,0.111248,0.114806,{'classifier__learning_rate': 0.02058531211006...
2,RandomForest,0.089167,0.101219,0.119613,"{'classifier__bootstrap': True, 'classifier__c..."
1,LogisticRegressionLasso,0.086928,0.085225,0.105378,"{'classifier__C': 0.4524488502720415, 'classif..."
3,XGBoost,0.083688,0.092975,0.082462,{'classifier__colsample_bytree': 0.85158322271...


In [118]:
df_results.to_csv(SAVE_RESULT, index=False)